# state-dict-load — worked example 3: Load backbone weights, keep a fresh head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `state-dict-load`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Transfer learning loads a pretrained backbone into a model whose classification head is freshly initialized for a new task. Using `strict=False` with a checkpoint containing only backbone keys loads the backbone and leaves the head untouched, reported as missing keys.

## Worked solution

We make a two-part model (`features` + `head`) and a checkpoint holding only the `features.*` weights from pretraining. After `load_state_dict(ckpt, strict=False)`, the features are overwritten with pretrained values while the head keeps its random init (it appears in `missing_keys`). We verify the features now equal the checkpoint and that exactly the head keys are reported missing. We print the missing keys and the features-match check.

In [ ]:
import torch.nn as nn


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Linear(8, 12)
        self.head = nn.Linear(12, 4)
    def forward(self, x):
        return self.head(self.features(x))


t.manual_seed(0)
model = Net()
pretrained = {
    'features.weight': t.randn(12, 8),
    'features.bias': t.randn(12),
}
result = model.load_state_dict(pretrained, strict=False)
print('missing (head):', sorted(result.missing_keys))
print('features loaded:', t.equal(model.features.weight.data, pretrained['features.weight']))